In [20]:
import pandas as pd
import numpy as np

In [21]:
# Definice cesty a načtení dat
file_path = r'C:\Users\Vít Dohnal\Downloads\ICOS_FR-Fon_FLUXNET_2019-2026_v1.3_r1\ICOS_FR-Fon_FLUXNET_FLUXMET_DD_2019-2026_v1.3_r1.csv'
sitename = ''

# Načtení dat s automatickým převedením FLUXNET chybějících hodnot na np.nan
df_raw = pd.read_csv(file_path, na_values=['-9999', '-9999.0'], parse_dates=['TIMESTAMP'])

In [22]:
# Dynamická detekce a ošetření chybějícího CO2
if 'CO2_F' in df_raw.columns:
    co2_series = df_raw['CO2_F']
elif 'CO2' in df_raw.columns:
    # Použití surových dat s interpolací krátkých výpadků (např. do 3 dnů)
    print("Upozornění: Používám surová data 'CO2' (gap-filled CO2_F není k dispozici).")
    co2_series = df_raw['CO2'].interpolate(method='linear', limit=3)
else:
    # Fallback pro klimatické modelování: fixní globální průměr pro dané období
    print("Upozornění: Lokální měření CO2 chybí. Aplikuji fixní hodnotu pozadí (400 ppm).")
    co2_series = 400.0

Upozornění: Lokální měření CO2 chybí. Aplikuji fixní hodnotu pozadí (400 ppm).


In [23]:
# Sestavení forcing dataframe pro rsofun
df_forcing = pd.DataFrame({
    'sitename': sitename,
    'date': df_raw['TIMESTAMP'],
    'temp': df_raw['TA_F'],                  
    'vpd': df_raw['VPD_F'] * 100,            
    'patm': df_raw['PA_F'] * 1000,           
    'ppfd': df_raw['SW_IN_F'] * 0.46 * 0.0864, 
    'rain': df_raw['P_F'],                   
    'snow': 0.0,                             
    'co2': co2_series                   
})

In [24]:
# Dodatečné ošetření chybějících hodnot (např. neúspěšně interpolované surové CO2)
# Nahrazení zbylých NaN v CO2 globálním průměrem
if isinstance(co2_series, pd.Series):
    df_forcing['co2'] = df_forcing['co2'].fillna(400.0)

# Filtrace dnů, kde chybí základní meteorologie (na těch model běžet nemůže)
df_forcing_clean = df_forcing.dropna(subset=['temp', 'vpd', 'ppfd', 'rain'])

output_path = f'rsofun_forcing_{sitename}.parquet'
df_forcing_clean.to_parquet(output_path, index=False)
print(f"\nUloženo {len(df_forcing_clean)} kompletních řádků do: {output_path}")


Uloženo 2791 kompletních řádků do: rsofun_forcing_.parquet
